# 01 — Data Exploration & Preprocessing

This notebook implements the pipeline described in **Section 4** of the report:

1. Loads raw MedMCQA and MedQA from `data/raw/`.
2. Applies quality filtering (§4.2.1), deduplication (§4.2.2), and cross-dataset leakage check (§4.2.3).
3. Runs sanity checks (§4.2.4): length distribution, A/B/C/D balance, specialty distribution, OOV.
4. Reports consolidated corpus (§4.2.5) and descriptive statistics (§4.3).
5. Verifies the `max_length=1024` truncation policy (§4.4.4) and runs a manual inspection (§4.4.7).
7. Saves figures to `reports/figures/eda/` + metadata to `report/eda report/`.

**Prerequisite**: run `python -m src.data.extract` to populate `data/raw/`.

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import random
import hashlib
import json
import re
from pathlib import Path

import langdetect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer, set_seed

# Resolve project root from notebook location
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "eda"
REPORT_DIR = PROJECT_ROOT / "reports" / "eda report"

# Tokenizer used for length analysis and OOV detection.
# NOTE: the Gemma 3 tokenizer is gated on HuggingFace. The user must accept
# the license at https://huggingface.co/google/gemma-3-4b-it and authenticate
# with `huggingface-cli login` before running this notebook.
TOKENIZER_NAME = "google/gemma-3-4b-it"

# Reproducibility
SEED = 42

# Create directories if they do not exist
for d in (FIGURES_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Set seed for reproducibility

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
langdetect.DetectorFactory.seed = SEED

# Display
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

# Uncomment to verify project root
#print("Project root:", PROJECT_ROOT)

## 2. Load raw datasets

In [ ]:
medmcqa = load_from_disk(str(RAW_DIR / "medmcqa"))
medqa = load_from_disk(str(RAW_DIR / "medqa"))

print("MedMCQA:")
for split, ds in medmcqa.items():
    print(f"  {split}: {len(ds):,} examples")
print("\nMedQA:")
for split, ds in medqa.items():
    print(f"  {split}: {len(ds):,} examples")

In [ ]:
# Inspect schemas before committing to specific field names
print("=== MedMCQA train columns ===")
print(medmcqa["train"].column_names)
print("\nFirst MedMCQA example:")
print(json.dumps(dict(medmcqa["train"][0]), indent=2))

In [ ]:
print("=== MedQA test columns ===")
print(medqa["test"].column_names)
print("\nFirst MedQA example:")
print(json.dumps(dict(medqa["test"][0]), indent=2, default=str))

## 3. Convert to DataFrames

Datasets are converted to Pandas DataFrames for the preprocessing operations. The splits are concatenated and tagged the origin in `_split` to preserve the original partition through the pipeline.

In [ ]:
def to_dataframe(ds_dict: DatasetDict) -> pd.DataFrame:
    frames = []
    for split, ds in ds_dict.items():
        df = ds.to_pandas()
        df["_split"] = split
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


medmcqa_df = to_dataframe(medmcqa)
medqa_df = to_dataframe(medqa)

print("MedMCQA combined:")
print(medmcqa_df["_split"].value_counts())
print(f"\nTotal: {len(medmcqa_df):,}")
print("\nMedQA combined:")
print(medqa_df["_split"].value_counts())
print(f"\nTotal: {len(medqa_df):,}")

## 4. Preprocessing

Track corpus attrition through every preprocessing stage. The dictionary is filled in along the way and used to render the consolidated attrition table at the end (§4.2.5).

In [ ]:
attrition: dict[str, dict[str, int]] = {
    "MedMCQA": {"raw": len(medmcqa_df)},
    "MedQA":   {"raw": len(medqa_df)},
}
attrition

### 4.1 Quality filtering (§4.2.1)

Filters applied to both datasets, with the `exp` (explanation) filter scoped to MedMCQA only — MedQA is used exclusively for evaluation, where the chain of reasoning is generated by the model rather than consumed from the dataset.

In [ ]:
def detect_lang_safe(text: str) -> str:
    try:
        return langdetect.detect(text)
    except langdetect.LangDetectException:
        return "unknown"


def _word_count(s: str) -> int:
    return len(str(s).split())

In [ ]:
def filter_medmcqa(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Apply the §4.2.1 quality filters to MedMCQA."""
    n_prev = len(df)
    if verbose:
        print(f"Initial: {n_prev:,}")

    # 1. non-empty required fields (question, options, explanation, label)
    required_text = ["question", "opa", "opb", "opc", "opd", "exp", "subject_name"]
    mask = df["cop"].notna()
    for col in required_text:
        mask &= df[col].notna() & (df[col].astype(str).str.strip().str.len() > 0)
    df = df[mask]
    if verbose:
        print(f"  After non-empty fields:  {len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 2. cop in {0, 1, 2, 3}
    df = df[df["cop"].astype(int).isin([0, 1, 2, 3])]
    if verbose:
        print(f"  After cop in range:      {len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 3. question with at least 5 whitespace tokens (lightweight pre-tokenizer proxy)
    df = df[df["question"].astype(str).map(_word_count) >= 5]
    if verbose:
        print(f"  After question >= 5 toks:{len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 4. language == English
    df = df.copy()
    df["_lang"] = df["question"].astype(str).map(detect_lang_safe)
    df = df[df["_lang"] == "en"].drop(columns=["_lang"])
    if verbose:
        print(f"  After language == en:    {len(df):,}  (dropped {n_prev - len(df):,})")

    return df.reset_index(drop=True)


print("=== Filtering MedMCQA ===")
medmcqa_filt = filter_medmcqa(medmcqa_df)
attrition["MedMCQA"]["filtered"] = len(medmcqa_filt)

In [ ]:
def filter_medqa(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Apply the §4.2.1 quality filters to MedQA."""
    
    n_prev = len(df)
    if verbose:
        print(f"Initial: {n_prev:,}")

    # 1. non-empty question and options
    mask = df["question"].notna() & (df["question"].astype(str).str.strip().str.len() > 0)
    mask &= df["options"].notna()
    df = df[mask]
    if verbose:
        print(f"  After non-empty fields:  {len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 2. answer_idx in {A, B, C, D}
    df = df[df["answer_idx"].astype(str).str.upper().isin(["A", "B", "C", "D"])]
    if verbose:
        print(f"  After answer_idx in set: {len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 3. question >= 5 whitespace tokens
    df = df[df["question"].astype(str).map(_word_count) >= 5]
    if verbose:
        print(f"  After question >= 5 toks:{len(df):,}  (dropped {n_prev - len(df):,})")
    n_prev = len(df)

    # 4. language == English
    df = df.copy()
    df["_lang"] = df["question"].astype(str).map(detect_lang_safe)
    df = df[df["_lang"] == "en"].drop(columns=["_lang"])
    if verbose:
        print(f"  After language == en:    {len(df):,}  (dropped {n_prev - len(df):,})")

    return df.reset_index(drop=True)


print("=== Filtering MedQA ===")
medqa_filt = filter_medqa(medqa_df)
attrition["MedQA"]["filtered"] = len(medqa_filt)

### 4.2 Deduplication (§4.2.2)

MD5 hash of the normalized question (lowercased, punctuation stripped, whitespace collapsed). Keep one copy of each unique question per dataset.

In [ ]:
_PUNCT_RE = re.compile(r"[^\w\s]")
_WS_RE = re.compile(r"\s+")


def normalize(text: str) -> str:
    text = str(text).lower()
    text = _PUNCT_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).strip()


def hash_question(text: str) -> str:
    return hashlib.md5(normalize(text).encode("utf-8")).hexdigest()


def dedup(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()
    df["_qhash"] = df["question"].astype(str).map(hash_question)
    n_before = len(df)
    df = df.drop_duplicates(subset="_qhash", keep="first").reset_index(drop=True)
    n_dups = n_before - len(df)
    print(f"{name}: {n_before:,} -> {len(df):,}  (duplicates removed: {n_dups:,})")
    return df


medmcqa_dedup = dedup(medmcqa_filt, "MedMCQA")
medqa_dedup = dedup(medqa_filt, "MedQA")

attrition["MedMCQA"]["deduplicated"] = len(medmcqa_dedup)
attrition["MedQA"]["deduplicated"] = len(medqa_dedup)

### 4.3 Cross-dataset leakage check (§4.2.3)

Any overlap between MedMCQA and MedQA test is removed from the **MedQA side**, prioritizing the integrity of the evaluation set.

In [ ]:
medmcqa_hashes = set(medmcqa_dedup["_qhash"])
overlap_mask = medqa_dedup["_qhash"].isin(medmcqa_hashes)
n_overlap = int(overlap_mask.sum())
print(f"Overlaps MedMCQA n MedQA test: {n_overlap}")

medqa_clean = medqa_dedup[~overlap_mask].reset_index(drop=True)
medmcqa_clean = medmcqa_dedup.copy()

attrition["MedMCQA"]["no_leakage"] = len(medmcqa_clean)
attrition["MedQA"]["no_leakage"] = len(medqa_clean)

print(f"MedMCQA after leakage check: {len(medmcqa_clean):,}")
print(f"MedQA after leakage check:   {len(medqa_clean):,}")

### 4.4 Sanity checks (§4.2.4)

Load the Gemma 3 tokenizer for length analysis and OOV detection.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Tokenizer: {TOKENIZER_NAME}")
print(f"Vocab size: {len(tokenizer):,}")
print(f"Special tokens: bos={tokenizer.bos_token!r}, eos={tokenizer.eos_token!r}, "
      f"unk={tokenizer.unk_token!r}, pad={tokenizer.pad_token!r}")

#### Prompt rendering

Render full chat-formatted examples using the §4.4 templates so token counts match what the training pipeline will see.

In [ ]:
USER_TEMPLATE = (
    "You are a medical expert. Answer the following multiple-choice question "
    "by reasoning step by step, then giving your final answer as a single letter.\n\n"
    "Question: {question}\n\n"
    "A) {opa}\n"
    "B) {opb}\n"
    "C) {opc}\n"
    "D) {opd}\n\n"
    "Reason step by step, then provide your answer in the format \"Answer: <letter>\"."
)

ASSISTANT_TEMPLATE = "Reasoning: {exp}\n\nAnswer: {letter}"

_LETTERS = ["A", "B", "C", "D"]


def render_medmcqa_full(row) -> str:
    """Render the full chat-formatted training example for a MedMCQA row."""
    user = USER_TEMPLATE.format(
        question=row["question"],
        opa=row["opa"], opb=row["opb"], opc=row["opc"], opd=row["opd"],
    )
    assistant = ASSISTANT_TEMPLATE.format(
        exp=row["exp"], letter=_LETTERS[int(row["cop"])],
    )
    msgs = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)


def render_medqa_user(row) -> str:
    """Render the user-side prompt for a MedQA row (eval-only, no assistant content)."""
    opts = row["options"]
    user = USER_TEMPLATE.format(
        question=row["question"],
        opa=opts["A"], opb=opts["B"], opc=opts["C"], opd=opts["D"],
    )
    msgs = [{"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


# Smoke test on one example each
print("=== MedMCQA formatted example ===")
print(render_medmcqa_full(medmcqa_clean.iloc[0]))
print("\n=== MedQA formatted example ===")
print(render_medqa_user(medqa_clean.iloc[0]))

#### Batch tokenization with length and OOV stats

Tokenize all examples in batches, recording token lengths and counting any `<unk>` occurrences in a single pass.

In [ ]:
def batch_tokenize_with_stats(
    texts: list[str],
    batch_size: int = 64,
    unk_id: int | None = None,
) -> tuple[list[int], int, int]:
    """Tokenize `texts` in batches and return (lengths, total_tokens, unk_count)."""
    lengths: list[int] = []
    total = 0
    unk = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(batch, add_special_tokens=False, padding=False)
        for ids in encoded["input_ids"]:
            lengths.append(len(ids))
            total += len(ids)
            if unk_id is not None:
                unk += sum(1 for tok_id in ids if tok_id == unk_id)
    return lengths, total, unk


unk_id = tokenizer.unk_token_id

print("Rendering and tokenizing MedMCQA (this may take a couple of minutes)...")
medmcqa_texts = [render_medmcqa_full(row) for _, row in medmcqa_clean.iterrows()]
medmcqa_lengths, medmcqa_total_tokens, medmcqa_unk = batch_tokenize_with_stats(
    medmcqa_texts, batch_size=64, unk_id=unk_id,
)
medmcqa_clean["_n_tokens"] = medmcqa_lengths

print("Rendering and tokenizing MedQA...")
medqa_texts = [render_medqa_user(row) for _, row in medqa_clean.iterrows()]
medqa_lengths, medqa_total_tokens, medqa_unk = batch_tokenize_with_stats(
    medqa_texts, batch_size=64, unk_id=unk_id,
)
medqa_clean["_n_tokens"] = medqa_lengths

print("Done.")

#### Length distribution (Figure 4.1)

In [ ]:
def length_percentiles(lengths: list[int], label: str) -> dict:
    arr = np.array(lengths)
    return {
        "dataset": label,
        "p50": int(np.percentile(arr, 50)),
        "p90": int(np.percentile(arr, 90)),
        "p95": int(np.percentile(arr, 95)),
        "p99": int(np.percentile(arr, 99)),
        "max": int(arr.max()),
    }


length_table = pd.DataFrame([
    length_percentiles(medmcqa_lengths, "MedMCQA (full)"),
    length_percentiles(medqa_lengths, "MedQA (user-only)"),
])
print(length_table.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(medmcqa_lengths, bins=60, alpha=0.6, density=True,
        label=f"MedMCQA full prompt+response (n={len(medmcqa_lengths):,})",
        color="steelblue")
ax.hist(medqa_lengths, bins=60, alpha=0.6, density=True,
        label=f"MedQA user prompt only (n={len(medqa_lengths):,})",
        color="darkorange")
ax.axvline(1024, color="red", linestyle="--", label="max_length = 1024")
ax.set_xlabel("Tokens (Gemma 3 tokenizer)")
ax.set_ylabel("Frequency")
ax.set_title("Length distribution of formatted examples")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "length_distribution.png", dpi=150)
plt.show()

#### A/B/C/D balance (Figure 4.2)

In [ ]:
medmcqa_letter_counts = (
    medmcqa_clean["cop"].astype(int)
    .map(lambda i: _LETTERS[i])
    .value_counts()
    .reindex(_LETTERS, fill_value=0)
)
medqa_letter_counts = (
    medqa_clean["answer_idx"].astype(str).str.upper()
    .value_counts()
    .reindex(_LETTERS, fill_value=0)
)

balance_df = pd.DataFrame({
    "MedMCQA": medmcqa_letter_counts,
    "MedQA":   medqa_letter_counts,
})
balance_pct = balance_df.div(balance_df.sum(axis=0), axis=1) * 100
print("Counts:")
print(balance_df)
print("\nPercentages:")
print(balance_pct.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
balance_pct.plot(kind="bar", ax=ax, color=["steelblue", "darkorange"])
ax.axhline(25, color="gray", linestyle="--", label="Uniform (25%)")
ax.set_xlabel("Correct answer letter")
ax.set_ylabel("Percentage")
ax.set_title("Distribution of correct answer letters")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "letter_balance.png", dpi=150)
plt.show()

#### Specialty distribution in MedMCQA (Figure 4.3)

MedQA does not provide a specialty field (see report §4.1) and is therefore not included here.

In [ ]:
specialty_counts = medmcqa_clean["subject_name"].value_counts()
print(f"Unique specialties: {len(specialty_counts)}")
print(specialty_counts)

fig, ax = plt.subplots(figsize=(10, 7))
specialty_counts.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Specialty")
ax.set_title("Specialty distribution in cleaned MedMCQA")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "medmcqa_specialty_distribution.png", dpi=150)
plt.show()

#### Out-of-vocabulary tokens

Gemma 3 uses a 256ki SentencePiece vocabulary trained on a massive multilingual corpus, so the expected OOV rate is effectively zero. The counts below come from the same single-pass tokenization used for the length distribution above.

In [ ]:
oov_table = pd.DataFrame({
    "Metric": ["Total tokens", "<unk> tokens", "% OOV"],
    "MedMCQA": [
        f"{medmcqa_total_tokens:,}",
        f"{medmcqa_unk:,}",
        f"{100 * medmcqa_unk / max(1, medmcqa_total_tokens):.4f}%",
    ],
    "MedQA": [
        f"{medqa_total_tokens:,}",
        f"{medqa_unk:,}",
        f"{100 * medqa_unk / max(1, medqa_total_tokens):.4f}%",
    ],
})
print(oov_table.to_string(index=False))

# OOV does not typically remove examples in modern SentencePiece tokenizers;
# preserve the same counts as the leakage-cleaned step.
attrition["MedMCQA"]["after_oov"] = len(medmcqa_clean)
attrition["MedQA"]["after_oov"] = len(medqa_clean)

### 4.5 Preprocessing summary (§4.2.5)

In [ ]:
stages = ["raw", "filtered", "deduplicated", "no_leakage", "after_oov"]
stage_labels = {
    "raw": "Raw corpus",
    "filtered": "After quality filtering",
    "deduplicated": "After deduplication",
    "no_leakage": "After cross-dataset leakage removal",
    "after_oov": "After OOV removal (if applicable)",
}

rows = []
for stage in stages:
    rows.append({
        "Stage": stage_labels[stage],
        "MedMCQA": attrition["MedMCQA"].get(stage, 0),
        "MedQA": attrition["MedQA"].get(stage, 0),
    })

rows.append({
    "Stage": "Total loss vs. raw (%)",
    "MedMCQA": f"{100 * (1 - attrition['MedMCQA']['after_oov'] / attrition['MedMCQA']['raw']):.2f}%",
    "MedQA":   f"{100 * (1 - attrition['MedQA']['after_oov']   / attrition['MedQA']['raw']):.2f}%",
})

attrition_table = pd.DataFrame(rows)
print(attrition_table.to_string(index=False))

## 5. Descriptive statistics (§4.3)

Per-field token lengths computed in batches over the full cleaned corpora.

In [ ]:
def batch_tokenize_lengths(texts: list[str], batch_size: int = 128) -> list[int]:
    out: list[int] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(batch, add_special_tokens=False, padding=False)
        out.extend(len(ids) for ids in encoded["input_ids"])
    return out


# MedMCQA per-field stats
mcqa_q_lens   = batch_tokenize_lengths(medmcqa_clean["question"].astype(str).tolist())
mcqa_opt_lens = batch_tokenize_lengths(
    medmcqa_clean[["opa", "opb", "opc", "opd"]].astype(str).values.flatten().tolist()
)
mcqa_exp_lens = batch_tokenize_lengths(medmcqa_clean["exp"].astype(str).tolist())

# MedQA per-field stats — options live inside a dict
qa_q_lens = batch_tokenize_lengths(medqa_clean["question"].astype(str).tolist())
qa_opt_texts = [opt for opts in medqa_clean["options"] for opt in opts.values()]
qa_opt_lens = batch_tokenize_lengths(qa_opt_texts)

stats_table = pd.DataFrame({
    "MedMCQA": {
        "Total questions": len(medmcqa_clean),
        "Mean question length (tokens)": float(np.mean(mcqa_q_lens)),
        "Mean option length (tokens)":  float(np.mean(mcqa_opt_lens)),
        "Mean explanation length (tokens)": float(np.mean(mcqa_exp_lens)),
        "Number of specialties": int(medmcqa_clean["subject_name"].nunique()),
    },
    "MedQA": {
        "Total questions": len(medqa_clean),
        "Mean question length (tokens)": float(np.mean(qa_q_lens)),
        "Mean option length (tokens)":  float(np.mean(qa_opt_lens)),
        "Mean explanation length (tokens)": None,
        "Number of specialties": None,
    },
})
print(stats_table.round(2))

## 6. Truncation policy verification (§4.4.4)

How many examples would be truncated under `max_length = 1024`? If the rate exceeds 5%, the ceiling should be reconsidered.

In [ ]:
TRUNCATION_THRESHOLD = 5 # percent
MAX_LENGTH = 1024
n_trunc_mcqa = int(np.sum(np.array(medmcqa_lengths) > MAX_LENGTH))
n_trunc_qa = int(np.sum(np.array(medqa_lengths) > MAX_LENGTH))

print(f"MedMCQA truncated: {n_trunc_mcqa:,} / {len(medmcqa_lengths):,} "
      f"({100 * n_trunc_mcqa / len(medmcqa_lengths):.2f}%)")
print(f"MedQA truncated:   {n_trunc_qa:,} / {len(medqa_lengths):,} "
      f"({100 * n_trunc_qa / len(medqa_lengths):.2f}%)")

if 100 * n_trunc_mcqa / len(medmcqa_lengths) > TRUNCATION_THRESHOLD:
    print("\nWARNING: truncation rate >10% on MedMCQA — consider raising max_length to 1280 or 1536.")
else:
    print(f"\nOK: truncation rate within target {TRUNCATION_THRESHOLD:.2f}%.")

## 7. Manual inspection (§4.4.7)

Random sample of 10 formatted examples for visual review.

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(medmcqa_clean), size=10, replace=False)
for i, idx in enumerate(sample_idx, 1):
    row = medmcqa_clean.iloc[idx]
    rendered = render_medmcqa_full(row)
    header = (f"Example {i}  (specialty={row['subject_name']}, "
              f"split={row['_split']})")
    print(f"\n{'=' * 80}\n{header}\n{'=' * 80}")
    print(rendered)

## 8. Generate report

In [ ]:
# Metadata file consolidating values referenced in the report.
# Also exposes the cleaned pool sizes so the splits configuration
# can be decided based on what is actually available.
metadata = {
    "seed": SEED,
    "tokenizer": TOKENIZER_NAME,
    "max_length": MAX_LENGTH,
    "attrition": attrition,
    "length_percentiles": length_table.to_dict(orient="records"),
    "letter_balance_pct": balance_pct.round(4).to_dict(),
    "specialty_counts": specialty_counts.to_dict(),
    "oov": {
        "medmcqa": {"total_tokens": medmcqa_total_tokens, "unk": medmcqa_unk},
        "medqa":   {"total_tokens": medqa_total_tokens,   "unk": medqa_unk},
    },
    "truncation": {
        "medmcqa": {"truncated": n_trunc_mcqa, "total": len(medmcqa_lengths)},
        "medqa":   {"truncated": n_trunc_qa,   "total": len(medqa_lengths)},
    },
    "descriptive_stats": stats_table.round(2).to_dict(),
    # Final pool sizes after preprocessing, broken down by native dataset
    "pool_sizes": {
        "medmcqa_train":      int((medmcqa_clean["_split"] == "train").sum()),
        "medmcqa_validation": int((medmcqa_clean["_split"] == "validation").sum()),
        "medqa_test":         int(len(medqa_clean)),
    },
}

metadata_path = REPORT_DIR / "metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
print(f"Metadata saved to {REPORT_DIR.relative_to(PROJECT_ROOT)}")

## Wrap-up

Outputs produced:

- `reports/figures/eda/length_distribution.png`
- `reports/figures/eda/letter_balance.png`
- `reports/figures/eda/medmcqa_specialty_distribution.png`
- `reports/eda report/metadata.json` — counts and metrics to report